In [1]:
import ollama
import numpy as np

In [2]:
EMBEDDING_MODEL = 'hf.co/CompendiumLabs/bge-base-en-v1.5-gguf'
LANGUAGE_MODEL = 'hf.co/bartowski/Llama-3.2-1B-Instruct-GGUF'

In [3]:
VECTOR_DB = []

In [4]:
def add_chunk_to_database(chunk):
    response = ollama.embed(model=EMBEDDING_MODEL, input=chunk)
    if 'embedding' in response:
        embedding = response['embedding']
    elif 'embeddings' in response:
        embedding = response['embeddings'][0]
    else:
        print(f"Chunk and vector adding failed")
        return  # jumps without any error proceeding to next step
    embedding_np = np.array(embedding, dtype='float')
    VECTOR_DB.append((chunk, embedding_np))
    print(f"Added chunks with embedding of size {embedding_np.shape}")

In [5]:
add_chunk_to_database("I'm not in Danger Skyler - I am the danger!.")

Added chunks with embedding of size (768,)


In [6]:
VECTOR_DB

[("I'm not in Danger Skyler - I am the danger!.",
  array([ 1.60938560e-02, -2.22745720e-02,  1.40641080e-02, -6.24927540e-02,
          3.79161700e-02,  2.73599100e-03,  7.18601300e-02, -2.42180470e-02,
          1.66098770e-02, -1.20914810e-02, -1.19290500e-02,  2.94672620e-02,
         -3.40733160e-02, -1.17185340e-02, -2.49380660e-02,  7.48669360e-02,
          3.19857900e-02,  2.22524500e-02,  2.62242240e-02, -1.87291630e-02,
          3.88344450e-02,  2.37917080e-04,  3.03919150e-03, -9.36358100e-03,
          5.04804500e-03, -5.28176700e-04, -2.06283090e-02,  3.86840780e-02,
         -1.52825810e-02,  2.70486230e-03,  5.18023860e-02, -1.87885200e-02,
         -1.48127140e-02, -1.48158020e-02,  3.82738540e-04,  4.69769160e-02,
         -3.50898100e-03,  1.63028110e-02, -2.05605220e-02, -2.62600460e-02,
         -3.68632380e-02, -9.42888100e-03, -1.77100000e-02, -3.02110540e-02,
         -4.31120500e-02, -5.45184760e-02,  1.81835600e-02, -1.95123350e-02,
          8.04161700e-03, 

In [7]:
# cosine similarity of a and b = (a . b) / |a|.|b|

In [8]:
def cosine_similarity(a,b):  # a is query vector b is vector in vectordb
    a_np = np.array(a, dtype='float')
    b_np = np.array(b, dtype='float')

    dot_product = np.dot(a_np,b_np)

    norm_a = np.linalg.norm(a_np)
    norm_b = np.linalg.norm(b_np)

    if norm_a == 0 or norm_b == 0:
        return 0.0
    else:
        return dot_product / (norm_a * norm_b)

In [9]:
def retrieve(query, top_n=3):
    try:
        response = ollama.embed(model=EMBEDDING_MODEL, input=query)
        if 'embeddings' in response:
            embeddings = response['embeddings']
            if isinstance(embeddings[0], list):
                query_embedding = embeddings[0]
            else:
                query_embedding = embeddings
        else:
            print(f"Unexpected response structure: {response.keys()}")
            raise ValueError("Could not find embedding in response")
        query_embedding_np = np.array(query_embedding, dtype='float')

        similarities = []
        for chunk, embedding_np in VECTOR_DB:
            similarity = cosine_similarity(query_embedding_np, embedding_np)
            similarities.append((chunk, similarity))
        similarities.sort(key=lambda x:x[1], reverse=True)

        return similarities[:top_n]
    
    except Exception as e:
        print(f"Error Retrieving : {e} ")

In [ ]:
def main():
    print("Loading dataset...")
    dataset = []
    with open('cat-facts.txt', 'r') as file:
        dataset = file.readlines()
    dataset = [line.strip() for line in dataset if line.strip()]
    print(f'Loaded {len(dataset)} entries')
    print('Indexing dataset (this might take a few mintues)...')
    for i, chunk in enumerate(dataset):
        add_chunk_to_database(chunk)
        print(f'Added chunk {i+1}/{len(dataset)} to the database')

    while True:
        input_query = input('\n Ask me a question about cats (or type "exit" to quit)')
        if input_query.lower() == 'exit':
            break:
        print('\n Retrieving knowledge')
        retrieved_knowledge = retrieve(input_query)
        print('\n Retrieved Information')
        for chunk, similarity in retrieved_knowledge:
            print(f" - (similarity : {similarity: .2}) {chunk}")

        instruction_prompt = f''' You are a helpful chatbot    

